# Semana 12 - Pré-processamento de Dados (Titanic)

**Objetivo:** Aplicar todas as etapas de pré-processamento em um
dataset real.

**Instruções:**
1. Baixar o notebook exemplo do repositório (dataset Titanic)
2. Executar o código completo – vejam se funciona
3. Testar as mudanças sugeridas no Slide 14
4. **Responder no notebook:**

* Qual estratégia deu o melhor resultado? Por quê?
* O que acontece se você NÃO escalonar os dados?
* O que acontece se você não tratar os valores nulos?

5. Subir o notebook respondido na pasta da semana 12 do repositório

# 1. Importar bibliotecas

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# 2. Carregar os dados

In [ ]:
# Carregar dataset Titanic
df = pd.read_csv('https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv')

# Visualizar primeiras linhas
df.head()

,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S
3,4,1,1,"Futrelle, Mrs. Jacques Heath (Lily May Peel)",female,35.0,1,0,113803,53.1000,C123,S
4,5,0,3,"Allen, Mr. William Henry",male,35.0,0,0,373450,8.0500,NaN,S


# 3. Separar X e y

In [ ]:
# Separar features (X) e target (y)
X = df.drop('Survived', axis=1)
y = df['Survived']

# Verificar formato
print(f'X: {X.shape}')
print(f'y: {y.shape}')

X: (891, 11)
y: (891,)


# 4. Definir colunas e criar pipelines

In [ ]:
# Definir colunas numéricas e categóricas
num_cols = ['Age', 'Fare']
cat_cols = ['Sex', 'Embarked']

# Pipeline para variáveis numéricas: imputar nulos (mediana) + escalonar (StandardScaler)
num_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Pipeline para variáveis categóricas: imputar nulos (mais frequente) + one-hot
cat_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

# Juntar tudo com ColumnTransformer
preprocessor = ColumnTransformer([
    ('num', num_pipeline, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

# 5. Criar Pipeline final

In [ ]:
# Pipeline completo: pré-processamento + modelo
pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', LogisticRegression(max_iter=1000))
])

# Dividir treino/teste
X_treino, X_teste, y_treino, y_teste = train_test_split(X, y, test_size=0.2, random_state=42)

# Treinar
pipeline.fit(X_treino, y_treino)

# Avaliar
y_pred = pipeline.predict(X_teste)
acuracia_base = accuracy_score(y_teste, y_pred)
print(f'Acurácia com StandardScaler + LogReg: {acuracia_base:.4f}')

Acurácia com StandardScaler + LogReg: 0.7765


# 6. Experimento 1: Mudar estratégia de imputação (mediana → média)

In [ ]:
# Mesmo pipeline, mas com média em vez de mediana
num_pipeline_mean = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler())
])

preprocessor_mean = ColumnTransformer([
    ('num', num_pipeline_mean, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

pipeline_mean = Pipeline([
    ('preprocessor', preprocessor_mean),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_mean.fit(X_treino, y_treino)
y_pred_mean = pipeline_mean.predict(X_teste)
acuracia_mean = accuracy_score(y_teste, y_pred_mean)

print(f'Acurácia com média (em vez de mediana): {acuracia_mean:.4f}')
print(f'Diferença: {acuracia_mean - acuracia_base:.4f}')

Acurácia com média (em vez de mediana): 0.7765
Diferença: 0.0000


# 7. Experimento 2: Trocar StandardScaler por MinMaxScaler

In [ ]:
# Pipeline com MinMaxScaler
num_pipeline_minmax = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', MinMaxScaler())
])

preprocessor_minmax = ColumnTransformer([
    ('num', num_pipeline_minmax, num_cols),
    ('cat', cat_pipeline, cat_cols)
])

pipeline_minmax = Pipeline([
    ('preprocessor', preprocessor_minmax),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_minmax.fit(X_treino, y_treino)
y_pred_minmax = pipeline_minmax.predict(X_teste)
acuracia_minmax = accuracy_score(y_teste, y_pred_minmax)

print(f'Acurácia com MinMaxScaler: {acuracia_minmax:.4f}')
print(f'Diferença: {acuracia_minmax - acuracia_base:.4f}')

Acurácia com MinMaxScaler: 0.7821
Diferença: 0.0056


# 8. Experimento 3: Trocar modelo (Árvore de Decisão)

In [ ]:
# Pipeline com DecisionTree (sem escalonamento, porque árvores não precisam)
num_pipeline_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline_tree = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_tree = ColumnTransformer([
    ('num', num_pipeline_tree, num_cols),
    ('cat', cat_pipeline_tree, cat_cols)
])

pipeline_tree = Pipeline([
    ('preprocessor', preprocessor_tree),
    ('classifier', DecisionTreeClassifier(random_state=42))
])

pipeline_tree.fit(X_treino, y_treino)
y_pred_tree = pipeline_tree.predict(X_teste)
acuracia_tree = accuracy_score(y_teste, y_pred_tree)

print(f'Acurácia com Decision Tree: {acuracia_tree:.4f}')
print(f'Diferença: {acuracia_tree - acuracia_base:.4f}')

Acurácia com Decision Tree: 0.7263
Diferença: -0.0503


# 9. Experimento 4: Ignorar escalonamento

In [ ]:
# Pipeline SEM escalonamento
num_pipeline_no_scale = Pipeline([
    ('imputer', SimpleImputer(strategy='median'))
])

cat_pipeline_no_scale = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor_no_scale = ColumnTransformer([
    ('num', num_pipeline_no_scale, num_cols),
    ('cat', cat_pipeline_no_scale, cat_cols)
])

pipeline_no_scale = Pipeline([
    ('preprocessor', preprocessor_no_scale),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_no_scale.fit(X_treino, y_treino)
y_pred_no_scale = pipeline_no_scale.predict(X_teste)
acuracia_no_scale = accuracy_score(y_teste, y_pred_no_scale)

print(f'Acurácia SEM escalonamento: {acuracia_no_scale:.4f}')
print(f'Diferença: {acuracia_no_scale - acuracia_base:.4f}')

Acurácia SEM escalonamento: 0.7765
Diferença: 0.0000


# 10. Experimento 5: Ignorar tratamento de nulos

In [ ]:
# Pipeline SEM tratamento de nulos (dropar nulos)
X_sem_nulos = X.dropna()
y_sem_nulos = y.loc[X_sem_nulos.index]

# Dividir
X_treino_no_nulos, X_teste_no_nulos, y_treino_no_nulos, y_teste_no_nulos = train_test_split(
    X_sem_nulos, y_sem_nulos, test_size=0.2, random_state=42
)

# Pipeline só com escalonamento e one-hot (sem imputer)
preprocessor_no_imputer = ColumnTransformer([
    ('num', StandardScaler(), num_cols),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_cols)
])

pipeline_no_imputer = Pipeline([
    ('preprocessor', preprocessor_no_imputer),
    ('classifier', LogisticRegression(max_iter=1000))
])

pipeline_no_imputer.fit(X_treino_no_nulos, y_treino_no_nulos)
y_pred_no_imputer = pipeline_no_imputer.predict(X_teste_no_nulos)
acuracia_no_imputer = accuracy_score(y_teste_no_nulos, y_pred_no_imputer)

print(f'Acurácia SEM tratamento de nulos (apenas drop): {acuracia_no_imputer:.4f}')
print(f'Diferença: {acuracia_no_imputer - acuracia_base:.4f}')

Acurácia SEM tratamento de nulos (apenas drop): 0.7297
Diferença: -0.0468


# 11. Comparação de Resultados

In [ ]:
# Tabela comparativa
resultados = pd.DataFrame({
    'Experimento': [
        'Base (mediana + StandardScaler + LogReg)',
        'Imputação com média',
        'MinMaxScaler',
        'Decision Tree',
        'Sem escalonamento',
        'Sem tratamento de nulos'
    ],
    'Acurácia': [
        acuracia_base,
        acuracia_mean,
        acuracia_minmax,
        acuracia_tree,
        acuracia_no_scale,
        acuracia_no_imputer
    ]
})

print('=== Resultados dos Experimentos ===')
print(resultados)

# Melhor resultado
melhor = resultados.loc[resultados['Acurácia'].idxmax()]
print(f'\n Melhor resultado: {melhor["Experimento"]} com acurácia de {melhor["Acurácia"]:.4f}')

=== Resultados dos Experimentos ===
                                Experimento  Acurácia
0  Base (mediana + StandardScaler + LogReg)  0.776536
1                       Imputação com média  0.776536
2                              MinMaxScaler  0.782123
3                             Decision Tree  0.726257
4                         Sem escalonamento  0.776536
5                   Sem tratamento de nulos  0.729730

 Melhor resultado: MinMaxScaler com acurácia de 0.7821


## Respostas das Perguntas - Semana 12

---

### 1. Qual estratégia deu o melhor resultado? Por quê?

O melhor resultado foi obtido com o **MinMaxScaler**, com acurácia de **0.7821**.

**Por quê?** O MinMaxScaler transforma os dados para ficarem entre 0 e 1, o que é especialmente útil quando os dados não seguem uma distribuição normal (que é o caso de `Age` e `Fare` no Titanic). A regressão logística é sensível à escala das variáveis, e o MinMaxScaler equilibrou bem a influência de `Age` (0-80) e `Fare` (0-512), resultando em um pequeno ganho de performance em relação ao StandardScaler (0.7765 → 0.7821).

---

### 2. O que acontece se você NÃO escalonar os dados?

Quando não escalonamos os dados, a acurácia ficou em **0.7765**, exatamente igual ao resultado da estratégia base com StandardScaler.

**Por quê?** Neste caso específico, a diferença foi nula porque a regressão logística, mesmo sendo sensível à escala, pode ter convergido para uma solução similar com ou sem escalonamento para este dataset. Isso não significa que escalonar é desnecessário — em muitos outros problemas, a diferença é significativa. O fato de não ter mudado aqui pode ser devido à natureza dos dados ou ao tamanho do dataset.

**Importante:** Mesmo que não tenha feito diferença aqui, escalonar continua sendo uma boa prática e deve ser feito em praticamente todos os projetos com modelos lineares.

---

### 3. O que acontece se você não tratar os valores nulos?

Quando simplesmente removemos as linhas com valores nulos (em vez de preenchê-los), a acurácia **caiu de 0.7765 para 0.7297**, uma queda de **-0.0468** (cerca de 4,7 pontos percentuais).

**Por quê?** Ao dropar linhas com nulos, perdemos uma parte significativa dos dados (principalmente da coluna `Age`, que tem muitos valores faltando). Menos dados para treinar → modelo menos robusto. Além disso, os dados que foram perdidos podem não ser aleatórios — pode haver um padrão (ex: pessoas mais velhas podem ter menos registros de idade). Isso introduz um viés no modelo.

**Conclusão:** A imputação (preencher com mediana ou média) é uma estratégia muito melhor do que simplesmente remover os dados faltantes, pois preserva a quantidade de dados e mantém a distribuição geral.

---

### 4. Qual modelo performou melhor: Regressão Logística ou Árvore de Decisão?

A **Regressão Logística** (com MinMaxScaler) performou **muito melhor** que a Árvore de Decisão.

| Modelo | Acurácia |
|--------|----------|
| Regressão Logística (MinMaxScaler) | **0.7821** |
| Decision Tree (sem ajuste) | 0.7263 |

**Diferença:** -0.0558 (queda de cerca de 5,6 pontos percentuais)

**Por quê?** A árvore de decisão, sem ajuste de hiperparâmetros, tende a **overfitar** — ou seja, ela decora os dados de treino e não generaliza bem para novos dados. Com o ajuste de hiperparâmetros (que veremos na semana 13), a árvore poderia performar melhor. Por enquanto, com os parâmetros padrão, a Regressão Logística é mais adequada para este dataset.

---

### 5. Tabela Comparativa Final

| Experimento | Acurácia | Diferença da Base |
|-------------|----------|-------------------|
| Base (mediana + StandardScaler + LogReg) | 0.7765 | — |
| Imputação com média | 0.7765 | 0.0000 |
| **MinMaxScaler** | **0.7821** | **+0.0056** |
| Decision Tree | 0.7263 | -0.0503 |
| Sem escalonamento | 0.7765 | 0.0000 |
| Sem tratamento de nulos | 0.7297 | -0.0468 |

---

### Conclusão

O pré-processamento adequado — especialmente o **tratamento de nulos** e o **escalonamento** — melhora (ou pelo menos mantém) o desempenho do modelo. Este exercício mostrou que:

1. **MinMaxScaler** foi ligeiramente melhor que StandardScaler neste dataset específico
2. **Ignorar o tratamento de nulos** causa a maior queda de performance (-4,7%)
3. A **Regressão Logística** com pré-processamento adequado superou a Árvore de Decisão sem ajustes
4. A **imputação com média** teve o mesmo resultado que a mediana, mostrando que para este dataset, ambas as estratégias funcionam de forma similar

> **Aprendizado principal:** O pré-processamento não é um "detalhe" — é uma etapa fundamental que pode impactar significativamente o resultado final. Nunca pule essa fase!